In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Calculate Euclidean Distance Minima and Maxima from Videos

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')


folder_path = '/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID].csv'
save_plot = True  # Set to True to save plots


def euclidean(x1, y1, x2, y2):
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)


def analyze_euclidean_distance(csv_file):
    try:
        # Load the CSV with multi-index columns
        df = pd.read_csv(csv_file, header=[1, 2], index_col=0)

        # Flatten the multi-index columns
        df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]

        # Extract coordinates for top and bottom points
        top_x, top_y = df['top_x'], df['top_y']
        bottom_x, bottom_y = df['bottom_x'], df['bottom_y']

        # Compute Euclidean distance for each frame
        euclid_distances = euclidean(top_x, top_y, bottom_x, bottom_y)

        min_euclid_dist = np.min(euclid_distances)
        max_euclid_dist = np.max(euclid_distances)

        return {
            "filename": os.path.basename(csv_file),
            "euclid_distances_per_frame": euclid_distances.tolist(), # Convert numpy array to list for easier handling
            "min_euclid_dist": min_euclid_dist,
            "max_euclid_dist": max_euclid_dist
        }

    except Exception as e:
        return {
            "filename": os.path.basename(csv_file),
            "euclid_distances_per_frame": [],
            "min_euclid_dist": np.nan,
            "max_euclid_dist": np.nan,
            "error": str(e)
        }

# === SORT HELPER ===
def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')

# === MAIN LOOP ===
analysis_results = []

# Sort all csv files in the folder based on video number
sorted_files = sorted(
    [f for f in os.listdir(folder_path) if f.endswith(".csv") and "DLC" in f],
    key=extract_number
)

# Process each csv file in the folder
for filename in sorted_files:
    file_path = os.path.join(folder_path, filename)
    result = analyze_euclidean_distance(file_path)
    analysis_results.append(result)


# === PRINT RESULTS ===
print("\n--- Euclidean Distance Analysis Results ---")
for res in analysis_results:
    print(f"File: {res['filename']}")
    if "error" in res:
        print(f"  Error: {res['error']}")
    else:
        print(f"  Min Euclidean Distance: {res['min_euclid_dist']:.2f}")
        print(f"  Max Euclidean Distance: {res['max_euclid_dist']:.2f}")
    print("-" * 30)

# === SAVE RESULTS ===
output_df = pd.DataFrame([
    {
        'filename': res['filename'],
        'min_euclid_dist': res['min_euclid_dist'],
        'max_euclid_dist': res['max_euclid_dist']
    }
    for res in analysis_results if 'error' not in res
])

# Extract subject line from folder_path (assuming folder_path is defined in a previous cell)
base_folder_name = os.path.basename(folder_path)
subject_line = re.sub(r'\s+csv$', '', base_folder_name).replace(' ', '-')

output_filename = f'/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID]/{subject_line}_euclid_dist_summary.csv'
output_df.to_csv(output_filename, index=False)

print(f"\nEuclidean distance summary saved to {output_filename}")
display(output_df.head())


In [ ]:
# Calculate Eye Aspect Ratio Minima and Maxima from Videos
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION ===
folder_path = '/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID].csv'  # path to folder with CSVs
save_plot = True  # Set to True to save plots


# === FUNCTIONS ===

def euclidean(x1, y1, x2, y2):
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def analyze_EAR(csv_file):
    try:
        # Load the CSV with multi-index columns
        df = pd.read_csv(csv_file, header=[1, 2], index_col=0)

        # Flatten the multi-index columns
        df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]

        # Extract coordinates
        p1_x, p1_y = df['p1_x'], df['p1_y']
        p2_x, p2_y = df['p2_x'], df['p2_y']
        p3_x, p3_y = df['p3_x'], df['p3_y']
        p4_x, p4_y = df['p4_x'], df['p4_y']
        p5_x, p5_y = df['p5_x'], df['p5_y']
        p6_x, p6_y = df['p6_x'], df['p6_y']
        top_x, top_y = df['top_x'], df['top_y']
        bottom_x, bottom_y = df['bottom_x'], df['bottom_y']

        # Calculate EAR and Euclidean distance
        vertical1 = euclidean(p2_x, p2_y, p6_x, p6_y)
        vertical2 = euclidean(p3_x, p3_y, p5_x, p5_y)
        horizontal = euclidean(p1_x, p1_y, p4_x, p4_y)
        ear = (vertical1 + vertical2) / (2.0 * horizontal)

        # Calculate min and max EAR
        min_ear = ear.min()
        max_ear = ear.max()

        return {
            "filename": os.path.basename(csv_file),
            "ear_per_frame": ear.tolist(), # Convert numpy array to list for easier handling
            "min_ear": min_ear,
            "max_ear": max_ear
        }

    except Exception as e:
        return {
            "filename": os.path.basename(csv_file),
            "ear_per_frame": [],
            "min_ear": np.nan,
            "max_ear": np.nan,
            "error": str(e)
        }


# === SORT HELPER ===
def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')


# === MAIN LOOP ===
analysis_results = []

# Sort all csv files in the folder based on video number
sorted_files = sorted(
    [f for f in os.listdir(folder_path) if f.endswith(".csv") and "DLC" in f],
    key=extract_number
)

# Classify blinks for each csv file in the folder
for filename in sorted_files:
    file_path = os.path.join(folder_path, filename)
    result = analyze_EAR(file_path)
    analysis_results.append(result)


# === PRINT RESULTS ===
print("\n--- EAR analysis results ---")
for res in analysis_results:
  print(f"File: {res['filename']}")
  if "error" in res:
    print(f"Error: {res['error']}")
  else:
    print(f"  Min EAR: {res['min_ear']:.2f}") # Changed to print min EAR
    print(f"  Max EAR: {res['max_ear']:.2f}") # Added to print max EAR
  print("-" *30)

# === SAVE RESULTS ===
output_df = pd.DataFrame([
    {
        'filename': res['filename'],
        'min_ear': res['min_ear'], # Changed from 'EAR' to 'min_ear'
        'max_ear': res['max_ear']  # Added 'max_ear'
    }
    for res in analysis_results if 'error' not in res
])

# Extract subject line from folder_path (assuming folder_path is defined in a previous cell)
base_folder_name = os.path.basename(folder_path)
subject_line = re.sub(r'\s+csv$', '', base_folder_name).replace(' ', '-')

output_filename = f'/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID]/{subject_line}_ear_summary.csv' # Changed filename to _ear_summary.csv
output_df.to_csv(output_filename, index=False)

print(f"\nEAR summary saved to {output_filename}")
display(output_df.head())

In [ ]:
# Calculate Inner Eye Angle (p1) Minima and Maxima from Videos

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION ===
folder_path = '/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID].csv'  # path to folder with CSVs
save_plot = True  # Set to True to save plots

# === ANGLE CALCULATION FUNCTION ===
def compute_single_angle(xa, ya, xb, yb, xc, yc):
    a = np.array([xa, ya]) - np.array([xb, yb])
    c = np.array([xc, yc]) - np.array([xb, yb])
    cos_theta = np.dot(a, c) / (np.linalg.norm(a) * np.linalg.norm(c))
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    angle_rad = np.arccos(cos_theta)
    return np.degrees(angle_rad)

def analyze_inner_eye_angle(csv_file):
    try:
        # Load the CSV with multi-index columns
        df = pd.read_csv(csv_file, header=[1, 2], index_col=0)

        # Flatten the multi-index columns
        df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]

        # Extract needed coordinates
        top_x, top_y = df['top_x'], df['top_y']
        bottom_x, bottom_y = df['bottom_x'], df['bottom_y']
        p1_x, p1_y = df['p1_x'], df['p1_y']  # Vertex point

        # Compute angles
        eyelid_angles = []
        for i in range(len(df)):
            try:
                angle = compute_single_angle(
                    top_x.iloc[i], top_y.iloc[i],
                    p1_x.iloc[i], p1_y.iloc[i],
                    bottom_x.iloc[i], bottom_y.iloc[i]
                )
            except:
                angle = np.nan
            eyelid_angles.append(angle)

        inner_eyelid_angles = pd.Series(eyelid_angles)

        min_p1 = np.min(inner_eyelid_angles)
        max_p1 = np.max(inner_eyelid_angles)

        return{
            "filename": os.path.basename(csv_file),
            "eyelid_angles_per_frame": inner_eyelid_angles.tolist(),
            "min_inner_eye_angle": min_p1,
            "max_inner_eye_angle": max_p1
        }

    except Exception as e:
        return{
          "filename": os.path.basename(csv_file),
            "eyelid_angles_per_frame": [],
            "min_inner_eye_angle": np.nan,
            "max_inner_eye_angle": np.nan,
            "error": str(e)
        }


# === SORT HELPER ===
def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')

# === MAIN LOOP ===
analysis_results = []

# Sort all csv files in the folder based on video number
sorted_files = sorted(
    [f for f in os.listdir(folder_path) if f.endswith(".csv") and "DLC" in f],
    key=extract_number
)

# Process each csv file in the folder
for filename in sorted_files:
    file_path = os.path.join(folder_path, filename)
    result = analyze_inner_eye_angle(file_path)
    analysis_results.append(result)


# === PRINT RESULTS ===
print("\n--- Inner Eye Angle (p1) Analysis Results ---")
for res in analysis_results:
    print(f"File: {res['filename']}")
    if "error" in res:
        print(f"  Error: {res['error']}")
    else:
        print(f"  Min Inner Eye Angle: {res['min_inner_eye_angle']:.2f}")
        print(f"  Max Inner Eye Angle: {res['max_inner_eye_angle']:.2f}")
    print("-" * 30)

# === SAVE RESULTS ===
output_df = pd.DataFrame([
    {
        'filename': res['filename'],
        'min_p1': res['min_inner_eye_angle'],
        'max_p1': res['max_inner_eye_angle']
    }
    for res in analysis_results if 'error' not in res
])

# Extract subject line from folder_path (assuming folder_path is defined in a previous cell)
base_folder_name = os.path.basename(folder_path)
subject_line = re.sub(r'\s+csv$', '', base_folder_name).replace(' ', '-')

output_filename = f'/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID]/{subject_line}_p1_summary.csv'
output_df.to_csv(output_filename, index=False)

print(f"\nInner Eye Angle summary saved to {output_filename}")
display(output_df.head())

In [ ]:
#Calculate Outer Eye Angle (p4) Minima and Maxima from Videos
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION ===
folder_path = '/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID].csv'  # path to folder with CSVs
save_plot = False  # Set to True to save plots

# === ANGLE CALCULATION ===
def compute_single_angle(xa, ya, xb, yb, xc, yc):
    a = np.array([xa, ya]) - np.array([xb, yb])
    c = np.array([xc, yc]) - np.array([xb, yb])
    cos_theta = np.dot(a, c) / (np.linalg.norm(a) * np.linalg.norm(c))
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    angle_rad = np.arccos(cos_theta)
    return np.degrees(angle_rad)


def analyze_outer_eye_angle(csv_file):
    try:
        # Load the CSV with multi-index columns
        df = pd.read_csv(csv_file, header=[1, 2], index_col=0)

        # Flatten the multi-index columns
        df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]

        # Extract required coordinates
        top_x, top_y = df['top_x'], df['top_y']
        bottom_x, bottom_y = df['bottom_x'], df['bottom_y']
        p4_x, p4_y = df['p4_x'], df['p4_y']  # Vertex point

        # Compute eyelid angle per frame
        eyelid_angles = []
        for i in range(len(df)):
            angle = compute_single_angle(
                    top_x.iloc[i], top_y.iloc[i],
                    p4_x.iloc[i], p4_y.iloc[i],
                    bottom_x.iloc[i], bottom_y.iloc[i])
            eyelid_angles.append(angle)

        eyelid_angles = pd.Series(eyelid_angles)

        min_p4 = np.min(eyelid_angles)
        max_p4 = np.max(eyelid_angles)

        return{
            "filename": os.path.basename(csv_file),
            "eyelid_angles_per_frame": eyelid_angles.tolist(),
            "min_outer_eye_angle": min_p4,
            "max_outer_eye_angle": max_p4
        }

    except Exception as e:
        return{
          "filename": os.path.basename(csv_file),
            "eyelid_angles_per_frame": [],
            "min_outer_eye_angle": np.nan,
            "max_outer_eye_angle": np.nan,
            "error": str(e)
        }


# === SORT HELPER ===
def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')

# === MAIN LOOP ===
analysis_results = []

# Sort all csv files in the folder based on video number
sorted_files = sorted(
    [f for f in os.listdir(folder_path) if f.endswith(".csv") and "DLC" in f],
    key=extract_number
)

# Process each csv file in the folder
for filename in sorted_files:
    file_path = os.path.join(folder_path, filename)
    result = analyze_outer_eye_angle(file_path)
    analysis_results.append(result)


# === PRINT RESULTS ===
print("\n--- Outer Eye Angle (p4) Analysis Results ---")
for res in analysis_results:
    print(f"File: {res['filename']}")
    if "error" in res:
        print(f"  Error: {res['error']}")
    else:
        print(f"  Min Outer Eye Angle: {res['min_outer_eye_angle']:.2f}")
        print(f"  Max Outer Eye Angle: {res['max_outer_eye_angle']:.2f}")
    print("-" * 30)

# === SAVE RESULTS ===
output_df = pd.DataFrame([
    {
        'filename': res['filename'],
        'min_p4': res['min_outer_eye_angle'],
        'max_p4': res['max_outer_eye_angle']
    }
    for res in analysis_results if 'error' not in res
])

# Extract subject line from folder_path (assuming folder_path is defined in a previous cell)
base_folder_name = os.path.basename(folder_path)
subject_line = re.sub(r'\s+csv$', '', base_folder_name).replace(' ', '-')

output_filename = f'/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID]/{subject_line}_p4_summary.csv'
output_df.to_csv(output_filename, index=False)

print(f"\nOuter Eye Angle summary saved to {output_filename}")
display(output_df.head())




In [ ]:
# Calculate Velocity Minima and Maxima from Videos

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION ===
folder_path = '/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID].csv'  # Path to folder with CSVs
fps = 30.0
save_plot = False  # Set to True to save plots

def analyze_velocity(csv_file):
    try:
        # Load the CSV with multi-index columns
        df = pd.read_csv(csv_file, header=[1, 2], index_col=0)

        # Flatten the multi-index columns
        df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]

        # Extract required coordinates
        top_x, top_y = df['top_x'], df['top_y']
        bottom_x, bottom_y = df['bottom_x'], df['bottom_y']

        # Calculate distance between top and bottom eyelid per frame
        eyelid_dist = np.sqrt((top_x - bottom_x)**2 + (top_y - bottom_y)**2)

        # Calculate Velocity = Δdistance * fps
        velocity = np.abs(np.diff(eyelid_dist)) * fps
        velocity = np.insert(velocity, 0, 0.0)

        min_velocity = np.min(velocity)
        max_velocity = np.max(velocity)

        return {
            "filename": os.path.basename(csv_file),
            "velocity_per_frame": velocity.tolist(), # Convert numpy array to list for easier handling
            "min_velocity": min_velocity,
            "max_velocity": max_velocity
        }

    except Exception as e:
        return {
            "filename": os.path.basename(csv_file),
            "velocity_per_frame": [],
            "min_velocity": np.nan,
            "max_velocity": np.nan,
            "error": str(e)
        }

# === SORT HELPER ===
def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')

# === MAIN LOOP ===
analysis_results = []

# Sort all csv files in the folder based on video number
sorted_files = sorted(
    [f for f in os.listdir(folder_path) if f.endswith(".csv") and "DLC" in f],
    key=extract_number
)

# Process each csv file in the folder
for filename in sorted_files:
    file_path = os.path.join(folder_path, filename)
    result = analyze_velocity(file_path)
    analysis_results.append(result)


# === PRINT RESULTS ===
print("\n--- Velocity Analysis Results ---")
for res in analysis_results:
    print(f"File: {res['filename']}")
    if "error" in res:
        print(f"  Error: {res['error']}")
    else:
        print(f"  Min Velocity: {res['min_velocity']:.2f}")
        print(f"  Max Velocity: {res['max_velocity']:.2f}")
    print("-" * 30)

# === SAVE RESULTS ===
output_df = pd.DataFrame([
    {
        'filename': res['filename'],
        'min_velocity': res['min_velocity'],
        'max_velocity': res['max_velocity']
    }
    for res in analysis_results if 'error' not in res
])

# Extract subject line from folder_path (assuming folder_path is defined in a previous cell)
base_folder_name = os.path.basename(folder_path)
subject_line = re.sub(r'\s+csv$', '', base_folder_name).replace(' ', '-')

output_filename = f'/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID]/{subject_line}_velocity_summary.csv'
output_df.to_csv(output_filename, index=False)

print(f"\nVelocity summary saved to {output_filename}")
display(output_df.head())


In [ ]:
# Calculate Eye Area Minima and Maxima from Videos

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from google.colab import drive
drive.mount('/content/drive')


folder_path = '/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID].csv'
save_plot = True  # Set to True to save plots


def polygon_area(x, y):
    """Calculate the area of a polygon using the shoelace formula."""
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


def analyze_eye_area(csv_file):
    try:
        # Load the CSV with multi-index columns
        df = pd.read_csv(csv_file, header=[1, 2], index_col=0)

        # Flatten the multi-index columns
        df.columns = ['{}_{}'.format(a, b) for a, b in df.columns]

        # Collect all 8 points
        points = ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'top', 'bottom']
        x_coords = np.stack([df[f"{pt}_x"] for pt in points], axis=1)
        y_coords = np.stack([df[f"{pt}_y"] for pt in points], axis=1)

        # Compute polygon area for each frame
        areas = np.array([polygon_area(x, y) for x, y in zip(x_coords, y_coords)])

        min_area = np.min(areas)
        max_area = np.max(areas)

        # Removed classification and associated plotting logic

        return {
            "filename": os.path.basename(csv_file),
            "areas_per_frame": areas.tolist(), # Convert numpy array to list for easier handling
            "min_eye_area": min_area,
            "max_eye_area": max_area
        }

    except Exception as e:
        return {
            "filename": os.path.basename(csv_file),
            "areas_per_frame": [],
            "min_eye_area": np.nan,
            "max_eye_area": np.nan,
            "error": str(e)
        }

# === SORT HELPER ===
def extract_number(filename):
    match = re.search(r'_(\d+)', filename)
    return int(match.group(1)) if match else float('inf')

# === MAIN LOOP ===
analysis_results = [] # Changed variable name from 'results'

# Sort all csv files in the folder based on video number
sorted_files = sorted(
    [f for f in os.listdir(folder_path) if f.endswith(".csv") and "DLC" in f],
    key=extract_number
)

# Process each csv file in the folder
for filename in sorted_files:
    file_path = os.path.join(folder_path, filename)
    result = analyze_eye_area(file_path) # Call the new function
    analysis_results.append(result)


# === PRINT RESULTS ===
print("\n--- Eye Area Analysis Results ---")
for res in analysis_results:
    print(f"File: {res['filename']}")
    if "error" in res:
        print(f"  Error: {res['error']}")
    else:
        print(f"  Min Eye Area: {res['min_eye_area']:.2f}")
        print(f"  Max Eye Area: {res['max_eye_area']:.2f}")
        # Printing all areas per frame can be very verbose, so it's commented out.
        # print(f"  Areas per frame: {res['areas_per_frame']}")
    print("-" * 30)



# === SAVE RESULTS ===
output_df = pd.DataFrame([
    {
        'filename': res['filename'],
        'min_eye_area': res['min_eye_area'],
        'max_eye_area': res['max_eye_area']
    }
    for res in analysis_results if 'error' not in res
])

# Extract subject line from folder_path (assuming folder_path is defined in a previous cell)
base_folder_name = os.path.basename(folder_path)
subject_line = re.sub(r'\s+csv$', '', base_folder_name).replace(' ', '-')

output_filename = f'/content/drive/MyDrive/<DATA_ROOT>/[SUBJECT_ID]_[SESSION_ID]/{subject_line}_eye_area_summary.csv'
output_df.to_csv(output_filename, index=False)

print(f"Eye area summary saved to {output_filename}")
display(output_df.head())